<a href="https://colab.research.google.com/github/aqrlouhanjoauhan/PakePlus-Android-v2.1.5/blob/main/youtube_subtitle_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 🚀 YouTube Knowledge Base Cloud Extraction Engine { display-mode: "form" }
#@markdown 💡 **Click the Run button (▶) on the left to start downloading subtitles!**

import sys, os, re, shutil, json, subprocess, glob
from google.colab import output, files

TARGET_URL = ""

# ==============================================================
# 1. 运行瞬间弹出原生 Prompt 引导用户粘贴
# ==============================================================
js_prompt_code = """
(async () => {
    let clipboardText = "";
    try {
        const text = await navigator.clipboard.readText();
        if (text && (text.includes('youtube.com') || text.includes('youtu.be'))) {
            clipboardText = text;
        }
    } catch(e) {}

    const userEntered = prompt("🎯 Please paste your YouTube Channel or Video URL below (Ctrl+V / Cmd+V):", clipboardText);
    return { target_url: userEntered };
})();
"""

try:
    res = output.eval_js(js_prompt_code)
    if res and res.get('target_url') and res['target_url'].strip():
        TARGET_URL = res['target_url'].strip()
    else:
        TARGET_URL = ""
except Exception as e:
    TARGET_URL = ""

if not TARGET_URL:
    print("\n❌ Extraction Terminated: YouTube URL is empty or process was cancelled.")
    print("💡 Please click Run (▶) again and paste a valid YouTube link.")
    sys.exit(0)

print(f"\n🎯 Target URL Confirmed: {TARGET_URL}")
print("⏳ Preparing cloud environment (takes ~10 seconds on first run)...")
os.system("pip install -q --upgrade youtube-transcript-api yt-dlp")
from youtube_transcript_api import YouTubeTranscriptApi

base_dir = "./transcripts_temp"
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir, exist_ok=True)

# 字幕文本清洗：彻底过滤 WEBVTT / 时间戳 / HTML / 弹幕 JSON
def clean_subtitle_text(text):
    if not text:
        return ""
    if text.strip().startswith("{") and ("replayChatItemAction" in text or "liveChat" in text or "clickTrackingParams" in text):
        return ""

    lines = text.splitlines()
    cleaned = []
    for line in lines:
        line = line.strip()
        if not line or "WEBVTT" in line or "Kind:" in line or "Language:" in line or "-->" in line:
            continue
        if re.match(r'^\d+$', line):
            continue
        line = re.sub(r'<[^>]+>', '', line)
        if not cleaned or cleaned[-1] != line:
            cleaned.append(line)
    return " ".join(cleaned)

# 🌟 超级万能字幕提取器：兼容所有中文变体 (zh, zh-Hans, zh-Hant, zh-CN, zh-TW 等)
def get_transcript_universal(vid):
    extracted = ""

    # --- 方法 1: YouTubeTranscriptApi (全语言智能穿透) ---
    try:
        t_list = YouTubeTranscriptApi.list_transcripts(vid)
        t_obj = None

        # 优先按优先级搜索所有可能出现的中文及英文标记
        priority_langs = ['zh-Hans', 'zh-CN', 'zh', 'zh-Hant', 'zh-TW', 'zh-HK', 'en']
        for lang in priority_langs:
            try:
                t_obj = t_list.find_transcript([lang])
                if t_obj: break
            except Exception:
                continue

        # 如果还是没匹配上，直接强行抓取列表里的第 1 个字幕（哪怕是其它语言）
        if not t_obj:
            try:
                t_obj = next(iter(t_list))
                # 如果是外语且支持翻译，自动翻译为简体中文
                if t_obj.is_translatable:
                    try: t_obj = t_obj.translate('zh-Hans')
                    except Exception: pass
            except Exception:
                pass

        if t_obj:
            fetched = t_obj.fetch()
            raw_text = " ".join([item.get('text', '') for item in fetched])
            extracted = clean_subtitle_text(raw_text)
    except Exception:
        pass

    # --- 方法 2: yt-dlp 全通配下载 (兜底) ---
    if not extracted.strip():
        try:
            v_url = f"https://www.youtube.com/watch?v={vid}"
            temp_vtt_prefix = f"/tmp/sub_{vid}"

            dl_cmd = [
                "yt-dlp", "--skip-download",
                "--write-sub", "--write-auto-sub",
                "--sub-langs", "zh.*,zh,en.*,all",   # 🌟 通配符匹配所有中文/英文
                "--sub-format", "vtt/srt/best",
                "--no-write-comments",
                "--output", f"{temp_vtt_prefix}.%(ext)s", v_url
            ]
            subprocess.run(dl_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

            downloaded_files = glob.glob(f"{temp_vtt_prefix}*")
            valid_sub_files = [f for f in downloaded_files if not f.endswith('.json')]
            if valid_sub_files:
                selected_file = valid_sub_files[0]
                for df in valid_sub_files:
                    if any(l in df.lower() for lang in ['zh', 'chinese', 'en'] for l in [lang]):
                        selected_file = df
                        break

                with open(selected_file, "r", encoding="utf-8", errors="ignore") as vf:
                    raw_vtt = vf.read()
                    extracted = clean_subtitle_text(raw_vtt)

            for df in downloaded_files:
                try: os.remove(df)
                except: pass
        except Exception:
            pass

    return extracted.strip()

# ==============================================================
# 2. 统一提取 Video ID / Playlist 架构
# ==============================================================
video_list = []
channel_title = "youtube_subtitles"

def extract_single_video_id(url):
    patterns = [
        r'(?:v=|\/)([0-9A-Za-z_-]{11}).*',
        r'youtu\.be\/([0-9A-Za-z_-]{11})',
        r'shorts\/([0-9A-Za-z_-]{11})'
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None

single_vid = extract_single_video_id(TARGET_URL)

if single_vid:
    print(f"\n🎬 Mode Detected: 【Single Video】(Extracted ID: {single_vid})")
    cmd = ["yt-dlp", "--dump-json", "--no-playlist", f"https://www.youtube.com/watch?v={single_vid}"]
    res = subprocess.run(cmd, capture_output=True, text=True)
    video_title = single_vid
    if res.returncode == 0:
        try:
            data = json.loads(res.stdout)
            video_title = data.get('title', single_vid)
        except Exception:
            pass

    video_list.append({'id': single_vid, 'title': video_title})
    safe_title = re.sub(r'[^\w\-_]', '_', video_title)[:30]
    channel_title = f"Video_{safe_title}"
else:
    print(f"\n📺 Mode Detected: 【Channel / Playlist Full Batch Download】")
    fetch_url = TARGET_URL.rstrip('/')
    if not any(fetch_url.endswith(sub) for sub in ['/videos', '/shorts', '/playlists']):
        fetch_url += '/videos'

    cmd = ["yt-dlp", "--flat-playlist", "--dump-single-json", fetch_url]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        data = json.loads(res.stdout)
        channel_title = re.sub(r'[^\w\-_]', '_', data.get('title', 'Channel'))
        for entry in data.get('entries', []):
            if entry.get('id') and entry.get('_type') != 'playlist':
                video_list.append({'id': entry['id'], 'title': entry.get('title', entry['id'])})

        total_found = len(video_list)
        print(f"✅ Successfully scanned ALL {total_found} real videos!")

# ==============================================================
# 3. 统一下载与提取字幕
# ==============================================================
BATCH_SIZE = 50

if not video_list:
    print("❌ Extraction Failed: Could not recognize video info. Please check the URL.")
else:
    print("\n📝 Extracting transcripts...")
    success_count = 0
    created_zips = []

    for idx, item in enumerate(video_list, 1):
        vid = item['id']
        title = item['title']

        batch_num = ((idx - 1) // BATCH_SIZE) + 1
        batch_dir = os.path.join(base_dir, f"part_{batch_num}")
        os.makedirs(batch_dir, exist_ok=True)

        safe_title = re.sub(r'[^\w\-_]', '_', title)[:40]
        filepath = os.path.join(batch_dir, f"{idx:03d}_[{vid}]_{safe_title}.txt")

        # 调用通用强用字幕提取器
        extracted_text = get_transcript_universal(vid)

        if extracted_text:
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(extracted_text)
            success_count += 1
            print(f"  └─ [{idx}/{len(video_list)}] ✅ Success: {title[:25]}...")
        else:
            print(f"  └─ [{idx}/{len(video_list)}] ⚠️ Skipped (No transcript): {title[:25]}...")

    if success_count > 0:
        print(f"\n📦 Packing finished! Successfully fetched {success_count} transcripts.")
        part_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]
        part_folders.sort()

        for folder in part_folders:
            folder_path = os.path.join(base_dir, folder)
            if os.listdir(folder_path):
                zip_name = f"{channel_title}_subtitles" if len(part_folders) == 1 else f"{channel_title}_subtitles_{folder}"
                zip_filepath = shutil.make_archive(zip_name, 'zip', folder_path)
                created_zips.append(zip_filepath)

        print(f"🚀 Triggering download for {len(created_zips)} ZIP file(s)...")
        for zip_file in created_zips:
            files.download(zip_file)
    else:
        print("\n❌ Extraction Failed: Selected video(s) contain no valid transcripts.")